In [1]:
import json
import pandas as pd
from tqdm import tqdm
from copy import deepcopy
from time import sleep

import numpy as np
from scipy.stats import kendalltau

from judges import MistralJudge
from utils import make_prompt

In [2]:
# Путь до данных в CSV формате, 
# например /home/jovyan/work/alexander_workspace/exp/mtsquad/results/MTS_answer_labeling_1_1.csv
data_path = "path to data in CSV format"
data_path = "/home/jovyan/work/alexander_workspace/exp/mtsquad/results/MTS_answer_labeling_1_1.csv"
# В случае, если необходимо разметить только часть строк в файле
# задается параметр nrows
data = pd.read_csv(data_path, nrows=150)

In [4]:
args = dict(
    client_args = dict(
        # путь до файла, где хранится ключ к API модели
        api_key = open("/home/jovyan/work/alexander_workspace/mistral_api_key", "r").read().strip()
    )
)
# в gen_args хранятся system, user, assistant промпты, а также некоторые параметры, передающиеся в API
gen_args = json.load(open("/home/jovyan/work/alexander_workspace/exp/mtsquad/configs/gen_args.json", "r"))

In [6]:
# В случае, если есть людская разметка, можно использовать ее в качестве таргета
data_input, data_target = data.drop(columns="Human assessment"), data["Human assessment"]

In [7]:
# В случае отсутствия человеческой разметки 
# data_input = data

In [8]:
judge = MistralJudge(args=args)

In [9]:
# Здесь будут храниться ответы модели-судьи
responses = []

In [10]:
# Следующая ячейка будет периодически падать из-за превышения лимита обращений по API
# Чтобы обработать все запросы, необходимо просто перезапускать ее, не перезапуская предыдущие ячейки

In [11]:
for ind, sample in tqdm(data_input.iloc[len(responses):].iterrows()):
    user_prompt = make_prompt(gen_args["user_prompt"], *sample)
    new_gen_args = deepcopy(gen_args)
    new_gen_args["user_prompt"] = user_prompt
    response = None
    num_retries = 0
    limit = 10
    while response is None or num_retries > limit:
        try:
            response = judge(new_gen_args)
        except Exception as e:
            print(e)
            sleep(10)
        num_retries += 1
    responses.append(response)

28it [01:35,  2.71s/it]

API error occurred: Status 429
{"message":"Requests rate limit exceeded"}
None


29it [01:46,  5.40s/it]

API error occurred: Status 429
{"message":"Requests rate limit exceeded"}
None


51it [03:12,  3.27s/it]

...

128it [09:43,  4.47s/it]

API error occurred: Status 429
{"message":"Requests rate limit exceeded"}
None


142it [10:57,  4.46s/it]

API error occurred: Status 429
{"message":"Requests rate limit exceeded"}
None


150it [11:41,  4.68s/it]


In [12]:
len(responses)

150

In [132]:
data["model_assessment"] = responses

In [133]:
save_path = "path to save responses"
data.to_csv(save_path, index=False)

Так как модель возвращает не только оценку, но и расуждения, необходимо извлечь оценку из ответа. Для этого реализован метод 

In [134]:
parsed_responses = []
for resp in responses:
    try:
        parsed_resp = judge._parse_score(resp)
    except:
        parsed_resp = -1
    parsed_responses.append(parsed_resp)

In [22]:
round((parsed_responses == data_target).mean(), 2)

NameError: name 'data_target' is not defined

In [74]:
corr = kendalltau(parsed_responses, data_target)
round(corr.statistic, 2)

0.17

In [135]:
parsed_responses = np.array(parsed_responses)

In [136]:
(parsed_responses == -1).sum()

27

In [137]:
bad_response_ids = parsed_responses == -1

In [138]:
fixed_responses = [1] * bad_response_ids.sum()

In [139]:
len(parsed_responses[bad_response_ids]) == len(fixed_responses)

True

In [140]:
[print(i, resp, end="\n" + "#" * 20 + "\n") for i, resp in enumerate(np.array(responses)[bad_response_ids])];

0 1

Объяснение: Сгенерированный ответ полностью соответствует ground truth ответу. Он называет тех же авторов, под влиянием которых слова песен Дилана становятся более литературными и суггестивными.
####################
1 Для оценки качества ответа на вопрос, давайте рассмотрим все элементы:

**Вопрос:** "В образе кого написан Платон на фреске Афинская школа?"

**Ground truth ответ:** "На фреске Афинская школа Платон написан в образе Леонардо да Винчи."

**Сгенерированный ответ:** "Платон написан в образе Леонардо да Винчи."

Оба ответа указывают, что Платон написан в образе Леонардо да Винчи, что отвечает на вопросительное слово "кого" из вопроса.

**Оценка модели:** 1
####################
2 Для оценки качества ответа на вопрос, давайте рассмотрим следующие элементы:

**Вопрос:** "Какая историческая личность написана Рафаэлем как автопортрет?"

**Ground truth ответ:** "Птолемей очень похож на автора фрески."

**Сгенерированный ответ:** "Платон написан в образе Леонардо да Винчи."

Дл

In [142]:
fixed_responses[2] = 0
fixed_responses[9] = 0
fixed_responses[12] = 0
fixed_responses[22] = 0

In [143]:
parsed_responses[bad_response_ids] = fixed_responses

In [83]:
(parsed_responses == data_target).mean()

0.92

In [84]:
round(kendalltau(parsed_responses, data_target).statistic, 2)

0.53

In [144]:
data["parsed_responses"] = parsed_responses

In [145]:
data.to_csv("MTS_answer_labeling_lmjudged_2_3.csv", index=False)

In [1]:
# data_target.mean()

In [1]:
from utils import parse_gen
import pandas as pd
from judges import MistralJudge
import json

pd.set_option('display.max_colwidth', None)

In [2]:
path = "mtsquad/results/MTS_answer_labeling_lmjudged_raw_1_3.csv"
data = pd.read_csv(path)

In [3]:
res_path = "mtsquad/results/MTS_answer_labeling_1_3.csv"
res = pd.read_csv(res_path)

In [4]:
endings = ["Оценка модели", "Оценка"]
scores = data.model_assessment.apply(lambda x: parse_gen(x, endings=endings))

In [5]:
(scores == -1).sum()

2

In [6]:
data[scores == -1]

,model_assessment
1013,"Похоже, что в последнем примере нет вопроса и сгенерированного ответа, а также ground truth ответа. Пожалуйста, предоставьте полную информацию, чтобы я могла правильно оценить ответ."
1014,"Извините, но в вашем запросе отсутствует вопрос, ground truth ответ и сгенерированный ответ, которые необходимы для оценки. Пожалуйста, предоставьте эти данные, чтобы я могла выполнить оценку."


In [7]:
res[(scores == -1).values]

,Question,Ground_truth_ans,Gen_answer,Human assessment
1013,"Пример, демонстрирующий оба типа изменений","Устойчивость бактерий к антибиотикам, или антибиотикорезистентность - пример, демонстрирующий оба типа изменений.","Устойчивость бактерий к антибиотикам, или антибиотикорезистентность.",NaN
1014,Спорная идея адаптации,"Спорная идея адаптации заключается в том, что некоторые адаптации могут повышать способность организмов к генетической изменяемости, повышая генетическое разнообразие, что в дальнейшем увеличивает вероятность адаптироваться в результате естественного отбора (повышение эволюционной способности организмов).",У меня нет ответа на ваш вопрос.,NaN


In [8]:
scores[scores == -1] = [1, 0]

In [9]:
round(scores.mean(), 3)

0.908

In [10]:
res["model_assessment"] = scores
res.to_csv(res_path, index=False)

In [112]:
results = {
    "exp_1_1": 0.928,
    "exp_1_2": 0.911,
    "exp_1_3": 0.908,
    "exp_2_1": 0.837,
    "exp_2_2": 0.761,
    "exp_2_3": 0.768,
    "exp_4_1": 0.906,
    "exp_4_2": 0.866,
    "exp_4_3": 0.888,
    "exp_5_1": 0.932,
    "exp_6_1": 0.220
}